In [ ]:
!nvidia-smi

NVIDIA-SMI has failed because it couldn't communicate with the NVIDIA driver. Make sure that the latest NVIDIA driver is installed and running.



In [ ]:
pip install --quiet scikit-optimize

     |████████████████████████████████| 81kB 2.2MB/s 


In [ ]:
%cd "/content/drive/My Drive/ICFernando/SEER Remodelado"

/content/drive/My Drive/ICFernando/SEER Remodelado


In [ ]:
import os
import sys
import pickle
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, r2_score
import skopt
from skopt.plots import plot_objective
from skopt.utils import use_named_args
from skopt.space import Real, Integer
from skopt import dump, load, plots, gp_minimize
PONTOS = 100

In [ ]:
space  = [Real(10**-3, 10**0, "log-uniform", name='learning_rate'),
          Real(0, 10, name='min_split_loss'),
          Integer(3, 25, name='max_depth'),
          Real(0, 10, name='min_child_weight'),
          Real(0, 10, name='max_delta_step'),
          Real(0.5, 1, name='subsample'),
          Real(0.1, 1, name='colsample_bytree'),
          Real(0.1, 1, name='colsample_bylevel'),
          Real(0.1, 1, name='colsample_bynode'),
          Real(0, 10, name='reg_lambda'),
          Real(0, 10, name='alpha'),]

In [ ]:
targets = ['CURADO', 'SOBRE_COM_CANCER', 'TEMPO_SOBRE']
tipos_tumor = ['OTHER', 'BREAST', 'MALEGEN', 'RESPIR', 'COLRECT', 'LYMYLEUK', 'DIGOTHR', 'URINARY', 'FEMGEN', 'ALL']

In [ ]:
@use_named_args(space)
def f(learning_rate, min_split_loss, max_depth, min_child_weight, max_delta_step,
      subsample, colsample_bytree, colsample_bylevel, colsample_bynode, reg_lambda, alpha):
    
    if target == 'TEMPO_SOBRE':
        objective = 'reg:squarederror'
        eval_metric = 'rmse'
    else:
        objective = 'binary:logistic'
        eval_metric = 'auc'
    params = dict(
        learning_rate=learning_rate,
        min_split_loss=min_split_loss,
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        max_delta_step=max_delta_step,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        colsample_bylevel=colsample_bylevel,
        colsample_bynode=colsample_bynode,
        reg_lambda=reg_lambda,
        alpha=alpha,
        sampling_method='gradient_based',
        random_state=0,
        n_jobs=-1,
        verbosity=0, 
        tree_method='gpu_hist',
        objective=objective, 
        booster='gbtree',
        eval_metric=eval_metric,
        )
    ml = xgb.train(
                params, dtrain, num_boost_round=100000,
                evals=[(dtrain, 'train'),
                       (dvalid, 'valid')],
                early_stopping_rounds=int(20),
                verbose_eval=10,
                )
    y_pred_scores = ml.predict(dtest, ntree_limit=ml.best_ntree_limit)
    
    if target == 'TEMPO_SOBRE':
        score = r2_score(y_test.values, np.round(y_pred_scores))
    else:
        score = roc_auc_score(y_test, y_pred_scores)
    return -score

In [ ]:
for target in targets:
    for tipo in tipos_tumor:
        print('-------------- ', target, tipo, ' --------------')

        if os.path.isfile('./bases/{0}/{1}/hiper_opt.pkl'.format(target.lower(), tipo)):
            res = load('./bases/{0}/{1}/hiper_opt.pkl'.format(target.lower(), tipo))
            PONTOS_ATUAL = PONTOS - len(res.x_iters)
            print(PONTOS_ATUAL)
            x0 = res.x_iters
            y0 = res.func_vals
            if PONTOS_ATUAL == 0:
                print('Pontos preenchidos, indo para o próximo problema...')
                continue
        else:
            PONTOS_ATUAL = PONTOS
            x0 = None
            y0 = None

        try:
            with open('./bases/{0}/{1}/X_train.pkl'.format(target.lower(), tipo), 'rb') as arq:
                X_train = pickle.load(arq)
                print('{0} carregado'.format('X_train'))
            with open('./bases/{0}/{1}/X_valid.pkl'.format(target.lower(), tipo), 'rb') as arq:
                X_valid = pickle.load(arq)
                print('{0} carregado'.format('X_valid'))
            with open('./bases/{0}/{1}/X_test.pkl'.format(target.lower(), tipo), 'rb') as arq:
                X_test = pickle.load(arq)
                print('{0} carregado'.format('X_test'))
            with open('./bases/{0}/{1}/y_train.pkl'.format(target.lower(), tipo), 'rb') as arq:
                y_train = pickle.load(arq)
                print('{0} carregado'.format('y_train'))
            with open('./bases/{0}/{1}/y_valid.pkl'.format(target.lower(), tipo), 'rb') as arq:
                y_valid = pickle.load(arq)
                print('{0} carregado'.format('y_valid'))
            with open('./bases/{0}/{1}/y_test.pkl'.format(target.lower(), tipo), 'rb') as arq:
                y_test = pickle.load(arq)
                print('{0} carregado'.format('y_test'))
        except OSError:
            print('Erro ao carregar os dados!')
            continue
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dvalid = xgb.DMatrix(X_valid, label=y_valid)
        dtest = xgb.DMatrix(X_test)

        checkpoint_callback = skopt.callbacks.CheckpointSaver('./bases/{0}/{1}/hiper_opt.pkl'.format(target.lower(), tipo))
        res = gp_minimize(f, space,  n_calls=PONTOS_ATUAL, callback=[checkpoint_callback], random_state=1, verbose=True, x0=x0, y0=y0)

--------------  CURADO OTHER  --------------


OSError: ignored